In [ ]:
# Colab setup: fetch the data and the helper file (skip if you run locally with the files here)
!wget -q https://raw.githubusercontent.com/lalcayaga/micromet-lecture6/main/osterild_fast_uvw.csv -O osterild_fast_uvw.csv
!wget -q https://raw.githubusercontent.com/lalcayaga/micromet-lecture6/main/l6_helpers.py -O l6_helpers.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from l6_helpers import *

case = "moderate"                       # "strong" in Bonus A
t, u = load_block(case, block=0)        # one 10-min block of u'(t), mean removed
U = U_MEAN[case]                        # mean wind speed (Lecture 5)
N = u.size                              # number of samples
T_rec = N * DT                          # record length (s)
print(f"{case}: U = {U} m/s, N = {N} samples, T_rec = {T_rec:.0f} s, var(u) = {np.var(u):.3f} m²/s²")

# Round 1 — The analyser, on real wind
### 46100 Micrometeorology · Lecture 6

In the game, the analyser found how loud each note was by **multiplying the signal by that note and adding up**.
On the slides that became

$$\hat u_l = \frac1T\int_0^T u'(t)\,e^{-i\omega_l t}\,dt, \qquad \omega_l = \frac{2\pi l}{T}.$$

Now you build that analyser yourself, run it on ten minutes of real Østerild wind, and find out how the FFT does
the same thing for all notes at once.

**Time: ~27' work + 3' pair answers.** Work in pairs (File → *Save a copy in Drive*, share it with your partner).

| Task | Time | What you'll do |
|---|---|---|
| 1 | 3' | Look at the wind and commit two guesses |
| 2 | 7' | The analyser for one note |
| 3 | 7' | The FFT: all notes at once |
| 4 | 6' | Do the faders add up to the variance? |
| 5 | 4' | From notes to eddy sizes in metres |

> **Getting stuck is part of this.** These tasks are designed so that you work it out; nobody expects you to
> know the answer beforehand. Stuck for 3 minutes? Open hint 1. **Call the teacher or TA** when you've opened two hints,
> or a check FAILS and you can't see why. Tell us what you expected and what you got.

## Task 1 — Look, then commit (3')
Before computing anything, commit two guesses in the cell below.
- Counting notes from the lowest up (l = 1, 2, 3, …), **how many do you need for 90 % of the variance?** (1 to 6000)
- In Lecture 5 you measured $L_u$ ≈ 136 m for this hour. **Which eddy size (m) do you expect to hold the most variance?**

In [ ]:
plt.figure(figsize=(11, 3)); plt.plot(t, u, lw=0.5)
plt.xlabel("t (s)"); plt.ylabel("u′ (m/s)"); plt.title(f"Østerild 103 m, {case} wind, one 10-min block"); plt.show()

my_guess_n90 = None        # number of notes for 90 % of the variance
my_guess_size_m = None     # eddy size (m) holding the most variance

## Task 2 — The analyser for one note (7')
`note` below is the note $e^{i\omega_l t}$, sampled at the same times as the wind.
1. Write `u_hat_l`: the analyser's reading for that note (one line, the formula above).
2. Run. A real wind needs the note *and* its mirror $-\omega_l$; together they make the piece $2\,\mathrm{Re}(\hat u_l e^{i\omega_l t})$,
   which is plotted on top of the wind.
3. Try several `l` (1, 3, 10, 100). Which removes the most variance?

In [ ]:
l = 3
w_l = 2 * np.pi * l / T_rec                 # angular frequency of note l (rad/s)
note = np.exp(1j * w_l * t)                 # the note, sampled

# your code here: the analyser's reading for this note
u_hat_l = None
assert u_hat_l is not None, "Write u_hat_l first."

piece = 2 * np.real(u_hat_l * note)         # note + its mirror -w_l
print(f"l = {l}: period {T_rec / l:.1f} s, |u_hat_l| = {abs(u_hat_l):.3f} m/s")
print(f"variance removed by this one piece: {1 - np.var(u - piece) / np.var(u):.1%}")
other = np.exp(1j * 2 * np.pi * (l + 1) * t / T_rec)
print(f"(check) analyser reading of note l+1 when fed note l: {abs(np.mean(note * np.conj(other))):.1e}  <- orthogonal")
plt.figure(figsize=(11, 3)); plt.plot(t, u, lw=0.5, label="u′"); plt.plot(t, piece, lw=2, label=f"note l = {l}")
plt.xlabel("t (s)"); plt.legend(); plt.show()

<details><summary><b>Hint 1</b></summary>

The formula is $(1/T)\int_0^T u'(t)\,e^{-i\omega_l t}\,dt$. With samples every Δt, what is $(1/T)\int \dots dt$?

</details>
<details><summary><b>Hint 2</b></summary>

$\frac{1}{T}\sum_j (\dots)\,\Delta t$ with $T = N\Delta t$ is just the **mean** of the samples. And $e^{-i\omega_l t}$ is `np.conj(note)`.

</details>
<details><summary><b>Hint 3</b></summary>

`u_hat_l = np.mean(u * np.conj(note))`

</details>

## Task 3 — The FFT: all notes at once (7')
Doing Task 2 for all 6000 notes is slow. `np.fft.fft` does the same multiply-and-add for every l in one go,
but with a different constant in front.
1. Write `X_fft`, the FFT of `u`. The cell compares `X_fft[l]` with your `u_hat_l` from Task 2.
2. From the ratio, work out the constant and write `u_hat`: the readings of *all* notes. The check must PASS.

In [ ]:
# your code here: the FFT of the whole record
X_fft = None
assert X_fft is not None, "Write X_fft first."
print(f"note l = {l}: FFT gives {X_fft[l]:.3f}, your analyser gave {u_hat_l:.5f}, ratio FFT/yours = {X_fft[l] / u_hat_l:.6g}")

# your code here: all the analyser readings at once, from X
u_hat = None
assert u_hat is not None, "Write u_hat first."
ok = np.isclose(u_hat[l], u_hat_l)
print(f"CHECK u_hat[{l}] = {u_hat[l]:.5f} vs your analyser {u_hat_l:.5f} -> {'PASS' if ok else 'FAIL'}")

<details><summary><b>Hint 1</b></summary>

Look at the ratio it printed. Is it a number you have already met in this notebook?

</details>
<details><summary><b>Hint 2</b></summary>

Compare it with `N` and with `T_rec`. `np.mean` divides by N; `np.fft.fft` divides by nothing.

</details>
<details><summary><b>Hint 3</b></summary>

`u_hat = X_fft / N`

</details>

*So the FFT is the analyser for all N notes at once, without the 1/N. The notes call this raw FFT $X_{FFT}$ (eq. 4.3.17);
their $X(\omega_l, T)$ (eq. 4.3.14) is the same sum with $\Delta t/2\pi$ in front: different constants, same idea.*

## Task 4 — Do the faders add up to the variance? (6')
Each note carries the variance $|\hat u_l|^2$ (the fader, squared).
1. **Commit first:** will the sum over all notes be *smaller than*, *equal to* or *larger than* `np.var(u)`?
2. Write one line: the sum. Then run: the plot also answers your Task 1 guess.

In [ ]:
my_guess_sum = None                   # "smaller", "equal" or "larger"

# your code here: the sum of |u_hat|^2 over all notes
total = None
assert total is not None and my_guess_sum is not None, "Commit a guess and write the total first."
print(f"your guess: {my_guess_sum}\nsum of the faders² = {total:.4f} m²/s²\nvariance np.var(u) = {np.var(u):.4f} m²/s²")

# each positive note l and its mirror -l together carry 2|u_hat_l|^2
share = 2 * np.abs(u_hat[1:N // 2]) ** 2 / np.var(u)
frac = np.cumsum(share)
n90 = np.argmax(frac >= 0.9) + 1
compare("notes for 90 % of the variance", my_guess_n90, n90)
plt.semilogx(np.arange(1, N // 2), frac); plt.axhline(0.9, c="k", ls="--")
plt.xlabel("notes kept (lowest first)"); plt.ylabel("share of var(u)"); plt.show()

<details><summary><b>Hint 1</b></summary>

Each note's share of the variance is $|\hat u_l|^2$: `np.abs(...)**2`.

</details>
<details><summary><b>Hint 2</b></summary>

Sum over **all** N entries of `u_hat`: the negative frequencies are stored in the second half of the array.

</details>
<details><summary><b>Hint 3</b></summary>

`total = np.sum(np.abs(u_hat) ** 2)`

</details>

*Why can't two different notes add cross terms? (Hint: the orthogonality check in Task 2, and Pythagoras.)* …

## Task 5 — From notes to eddy sizes (4')
Note l repeats every $T_{rec}/l$ seconds. Write the eddy size it stands for (Taylor: size = wind × period),
then compare the tallest bar with your guess and with $L_u$.

In [ ]:
l_pos = np.arange(1, N // 2)                     # positive notes
var_modes = 2 * np.abs(u_hat[1:N // 2]) ** 2      # variance of each note (with its mirror)

# your code here: the eddy size (m) of every positive note
lam_l = None
assert lam_l is not None, "Write lam_l first."

centres, band_var, edges = size_bands(lam_l, var_modes)
peak = centres[np.argmax(band_var)]
compare("eddy size holding the most variance (m)", my_guess_size_m, peak)
print(f"L_u (Lecture 5) = {L_U_LECTURE5[case]:.0f} m, ratio peak / L_u = {peak / L_U_LECTURE5[case]:.1f}")
plt.figure(figsize=(11, 3.5)); plt.bar(centres, band_var / np.var(u), width=np.diff(edges) * 0.85, color="#2F3EEA")
plt.xscale("log"); plt.gca().invert_xaxis(); plt.axvline(L_U_LECTURE5[case], c="#990000", ls="--", label="L_u")
plt.xlabel("eddy size (m)  ← big eddies left"); plt.ylabel("share of var(u)"); plt.legend(); plt.show()

<details><summary><b>Hint 1</b></summary>

Period of note l: $T_{rec}/l$. An eddy that takes that long to pass at speed U has size …?

</details>
<details><summary><b>Hint 2</b></summary>

size = U × period.

</details>
<details><summary><b>Hint 3</b></summary>

`lam_l = U * T_rec / l_pos`

</details>

---
### Your pair's answers (3') — for the report-out
1. How many notes for 90 % of the variance, and how far off was your guess?
2. What did the faders² add up to? Why is that useful?
3. Which eddy size holds the most variance? Is it $L_u$?

---
## Bonus (only if you finish early)
**A.** Set `case = "strong"` in the setup cell and rerun. Does the tallest bar move to bigger eddies, to higher notes, or both?
**B.** Use `block=3`. How much do the bars change between two blocks of the same hour?